In [3]:
# Import libraries used for DICOM metadata scanning and parallel processing.
import pydicom
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# Configure the source directory and series matching rule.
main_folder = Path(r"\\pom\pmcommon\_PM.Private\CSIC\Moulin\4D Flow Data")
TARGET_SERIES_DESCRIPTION = "4D"
MATCH_MODE = "contains"

# Inspect one patient folder and collect matching scan metadata.
def process_patient(patient_dir):
    matches = []
    found_any = False
    for sequence_dir in sorted(patient_dir.iterdir()):
        if not sequence_dir.is_dir():
            continue
        dcm_files = list(sequence_dir.glob("*.dcm"))
        if not dcm_files:
            continue
        try:
            ds = pydicom.dcmread(dcm_files[0], stop_before_pixels=True)
            series_desc = getattr(ds, "SeriesDescription", "") or ""
            match = (
                TARGET_SERIES_DESCRIPTION.lower() in series_desc.lower()
                if MATCH_MODE == "contains"
                else series_desc.lower() == TARGET_SERIES_DESCRIPTION.lower()
            )
            if match:
                matches.append({
                    "Patient Name": getattr(ds, "PatientName", None),
                    "patient_folder": patient_dir.name,
                    "sequence_folder": sequence_dir.name,
                    "scan_path": str(sequence_dir),
                    "StudyDate": getattr(ds, "StudyDate", None),
                    "num_files": len(dcm_files),
                    "PatientMRN": getattr(ds, "PatientID", None),
                    "SeriesDescription": series_desc,
                    "Modality": getattr(ds, "Modality", None),
                })
                found_any = True
        except Exception as e:
            print(f"Skipped {sequence_dir}: {e}")
    return matches, (patient_dir.name if not found_any else None)

# Find patient folders and scan them concurrently.
patient_dirs = [d for d in main_folder.iterdir() if d.is_dir()]
records = []
no_match = []

with ThreadPoolExecutor(max_workers=16) as executor:
    futures = {executor.submit(process_patient, pd_dir): pd_dir for pd_dir in patient_dirs}
    for future in tqdm(as_completed(futures), total=len(futures)):
        matches, unmatched_patient = future.result()
        records.extend(matches)
        if unmatched_patient:
            no_match.append(unmatched_patient)

# Build the metadata table and rename columns for display.
df = pd.DataFrame(records)
print(f"Found {len(df)} matching sequences")

df = df.rename(columns={
    'patient_folder': 'Accession Number',
    'sequence_folder': 'Sequence Folder',
    'num_files': 'Image Count',
    'SeriesDescription' : 'Series Description',
})

# Apply table styling for notebook display.
(df.style
 .set_properties(**{
     'text-align': 'center',
     'font-family': 'Arial, sans-serif',
     'border': '1px solid white'
 }))

# Sort the metadata and save it for the interactive viewer.
df["Patient Name"] = df["Patient Name"].astype(str)
df = df.sort_values("Patient Name").reset_index(drop=True)
df.to_csv("4dflow_scan_metadata.csv", index=False)
print(f"Saved {len(df)} rows to 4dflow_scan_metadata.csv")

100%|██████████| 141/141 [11:12<00:00,  4.77s/it]

Found 410 matching sequences
Saved 410 rows to 4dflow_scan_metadata.csv


In [4]:
# Enable interactive Matplotlib output for this notebook.
%matplotlib widget

In [5]:
# Load the saved scan metadata for the interactive viewer.
import pandas as pd
df_sorted = pd.read_csv("4dflow_scan_metadata.csv")
print(f"Loaded {len(df_sorted)} rows")

Loaded 410 rows


In [6]:
# Display Matplotlib figures directly in the notebook.
%matplotlib inline

In [ ]:
# Import widget, plotting, numerical, and canvas tools used by the viewer.
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from ipycanvas import Canvas, hold_canvas

# Prepare display data and delineate visible and hidden columns.
df_display = df_sorted.reset_index(drop=True)
df_display["Patient Name"] = df_display["Patient Name"].str.replace("^", ", ", regex=False)
patient_cols = ["Patient Name", "PatientMRN"]
hidden_cols = ["scan_path", "StudyDate"]
scan_cols = [c for c in df_display.columns if c not in patient_cols + hidden_cols]

# Create reusable widget layouts and cell formatting.
cell_layout = widgets.Layout(width="150px", overflow="hidden")
checkbox_layout = widgets.Layout(width="60px")
row_layout = widgets.Layout(width="100%", min_height="35px", height="35px", padding="4px 0px", overflow="hidden")

# Create a centered HTML table cell with optional bold text.
def centered(text, bold=False):
    weight = "font-weight:bold;" if bold else ""
    return widgets.HTML(value=f"<div style='text-align:center; {weight}' title='{text}'>{text}</div>", layout=cell_layout)

# Create and style the patient search box.
search_box = widgets.Text(
    placeholder="Search by patient name or MRN...",
    layout=widgets.Layout(width="300px", margin="20px 0px 20px 10px")
)
search_box.add_class("mytext")

display(HTML("""
<style>
.mytext > .widget-label { font-size: 20px; }
.mytext > input[type="text"] {
    font-family: 'Calibri', sans-serif;
    font-size: 16px;
    border: 2px solid #36454F !important;
    border-radius: 4px;
}
</style>
"""))

# Build table headers and the selected-scan summary panel.
# Build the column header row for the scan-selection table.
def build_header():
    return widgets.HBox(
        [widgets.HTML(value="<div style='text-align:center; font-weight:bold;'>Select</div>", layout=checkbox_layout)] +
        [centered(col, bold=True) for col in scan_cols],
        layout=widgets.Layout(width="100%", border="0px 0px 2px 0px solid black", padding="4px 0px")
    )

selected_list_container = widgets.VBox(
    [],
    layout=widgets.Layout(display="flex", flex_flow="column", width="100%")
)

# Refresh the list of scans currently selected by the user.
def update_selected_summary():
    selected_indices = [i for i, cb in checkboxes.items() if cb.value]
    if not selected_indices:
        selected_list_container.children = [widgets.HTML(value="<i>No scans selected.</i>")]
        return

    rows = []
    for i in selected_indices:
        row = df_display.loc[i]
        seq_label = row.get("Sequence Folder", row.get(scan_cols[0], ""))
        label_text = f"{row['Patient Name']} | {seq_label}"
        label = widgets.HTML(
            value=f"<span style='font-size:13px;'>{label_text}</span>",
            layout=widgets.Layout(width="300px", overflow="hidden")
        )
        remove_btn = widgets.Button(
            description="x",
            layout=widgets.Layout(width="28px", height="28px"),
            tooltip="Remove"
        )

        # Create a callback that unchecks one selected scan.
        def make_remove_handler(idx):
            def handler(b):
                checkboxes[idx].value = False
            return handler

        remove_btn.on_click(make_remove_handler(i))
        rows.append(widgets.HBox(
            [label, remove_btn],
            layout=widgets.Layout(align_items="center", padding="2px 0px", min_height="30px", flex="0 0 auto")
        ))

    selected_list_container.children = rows

# Group scans by patient and build the accordion rows.
checkboxes = {}
patient_sections = {}

# Format DICOM study dates for readable display.
def format_study_date(date_val):
    try:
        return datetime.strptime(str(date_val), "%Y%m%d").strftime("%m/%d/%Y")
    except (ValueError, TypeError):
        return str(date_val)

# Extract the numeric sequence prefix used for scan ordering.
def scan_sort_key(sequence_folder):
    try:
        return int(str(sequence_folder).split("_")[0])
    except (ValueError, IndexError):
        return float("inf")

for patient_key, group in df_display.groupby(patient_cols, sort=False):
    name, mrn = patient_key if isinstance(patient_key, tuple) else (patient_key, "")
    title = f"{name} | MRN: {mrn} ({len(group)} scans)"
    rows = [build_header()]
    for date_val, date_group in group.sort_values("StudyDate", ascending=False).groupby("StudyDate", sort=False):
        rows.append(widgets.HTML(
            value=f"<div style='font-weight:bold; padding:8px 0 2px;'>{format_study_date(date_val)}</div>"
        ))
        date_group_sorted = date_group.assign(
            _scan_num=date_group["Sequence Folder"].apply(scan_sort_key)
        ).sort_values("_scan_num")
        for i, row in date_group_sorted.iterrows():
            cb = widgets.Checkbox(value=False, indent=False, layout=checkbox_layout)
            cb.observe(lambda change: update_selected_summary(), names="value")
            checkboxes[i] = cb
            cells = [cb] + [centered(row[col]) for col in scan_cols]
            rows.append(widgets.HBox(cells, layout=row_layout))

    patient_sections[patient_key] = (title, widgets.VBox(rows))

update_selected_summary()

# Configure the accordion and search filtering.
accordion = widgets.Accordion(children=[body for _, body in patient_sections.values()])
for idx, (title, _) in enumerate(patient_sections.values()):
    accordion.set_title(idx, title)

accordion_container = widgets.VBox(
    [accordion],
    layout=widgets.Layout(height="400px", width="100%", overflow="auto", border="1px solid lightgray")
)

# Replace accordion contents with the patients matching the search.
def rebuild_accordion(filtered_keys):
    children = [patient_sections[k][1] for k in filtered_keys]
    accordion.children = children
    for idx, k in enumerate(filtered_keys):
        accordion.set_title(idx, patient_sections[k][0])

# Filter patient sections when the search box changes.
def on_search_change(change):
    query = change["new"].strip().lower()
    if not query:
        rebuild_accordion(list(patient_sections.keys()))
    else:
        mask = (
            df_display["Patient Name"].astype(str).str.lower().str.contains(query, na=False) |
            df_display["PatientMRN"].astype(str).str.lower().str.contains(query, na=False)
        )
        matching_patients = df_display.loc[mask, patient_cols].drop_duplicates()
        filtered_keys = [tuple(x) for x in matching_patients.itertuples(index=False)]
        rebuild_accordion([k for k in patient_sections.keys() if k in filtered_keys])

search_box.observe(on_search_change, names="value")

# Create the run controls and output area.
selected_container = widgets.VBox(
    [selected_list_container],
    layout=widgets.Layout(height="100px", width="100%", overflow="auto", border="1px solid lightgray", padding="6px")
)
run_button = widgets.Button(description="Run", button_style="primary")
output = widgets.Output()

# Convert phase pixels and render magnitude or velocity images for the canvases.
# Convert encoded phase pixel values into signed velocity values.
def phase_to_velocity(phase_pixels, venc, grayscale_mid=2048):
    return (phase_pixels.astype(float) - grayscale_mid) / grayscale_mid * venc

# Overlay velocity colors on a grayscale magnitude image.
# Blend velocity colors over a normalized magnitude image.
def render_to_rgba(mag_img, vel_img, venc, alpha=0.5):
    mag_norm = (mag_img - mag_img.min()) / (mag_img.max() - mag_img.min() + 1e-8)
    gray_rgba = cm.gray(mag_norm)

    vel_norm = np.clip((vel_img + venc) / (2 * venc), 0, 1)
    vel_rgba = cm.seismic(vel_norm)

    out = gray_rgba.copy()
    out[..., :3] = (1 - alpha) * gray_rgba[..., :3] + alpha * vel_rgba[..., :3]
    out[..., 3] = 1.0
    return (out * 255).astype(np.uint8)

# Convert a magnitude image to grayscale RGBA pixels.
# Render a magnitude image as grayscale RGBA pixels.
def render_mag_only(mag_img):
    mag_norm = (mag_img - mag_img.min()) / (mag_img.max() - mag_img.min() + 1e-8)
    return (cm.gray(mag_norm) * 255).astype(np.uint8)

# Compute an MRA image: (1/N) * sum over time of [ M^2 * (Vx^2 + Vy^2 + Vz^2) ].
# Calculate the time-averaged MRA image for one slice.
def compute_mra(scans, mag_arr, v_mag, direction_panels, venc, slice_idx):
    missing = [axis for axis in ["X", "Y", "Z"] if axis not in direction_panels]
    if missing:
        return None, missing

    n_time = mag_arr.shape[4]
    accum = np.zeros(mag_arr.shape[:2], dtype=float)

    for t in range(n_time):
        M = mag_arr[:, :, slice_idx, v_mag, t]
        v_sum_sq = np.zeros_like(M)
        for axis in ["X", "Y", "Z"]:
            phase_idx, v_phase = direction_panels[axis]
            phase_pixels = scans[phase_idx]["array"][:, :, slice_idx, v_phase, t]
            v = phase_to_velocity(phase_pixels, venc)
            v_sum_sq += v ** 2
        accum += (M ** 2) * v_sum_sq

    mra = accum / n_time
    return mra, None

# Convert an MRA image to displayable grayscale RGBA pixels.
# Normalize and threshold an MRA image for grayscale display.
def render_mra(mra_img, threshold=0.0):
    display_values = np.sqrt(np.clip(mra_img, 0, None))
    norm = (display_values - display_values.min()) / (
        display_values.max() - display_values.min() + 1e-8
    )
    rgba = (cm.gray(norm) * 255).astype(np.uint8)
    rgba[norm < threshold, :3] = 0
    return rgba

# Manage image display and line drawing for one interactive canvas.
class CanvasLineDrawer:
    """Wraps an ipycanvas Canvas: shows an image, lets the user click-drag-release to draw lines."""
    # Initialize the canvas and mouse handlers for line drawing.
    def __init__(self, image_rgba):
        h, w = image_rgba.shape[:2]
        self.canvas = Canvas(width=w, height=h)
        self.image_rgba = image_rgba
        self.lines = []  # (xmin, ymin, xmax, ymax) per completed line
        self.start = None
        self.on_line_drawn = None
        self._mra_missing = None  # set externally for the MRA panel when directions are missing

        self.draw_image()

        self.canvas.on_mouse_down(self.on_press)
        self.canvas.on_mouse_move(self.on_motion)
        self.canvas.on_mouse_up(self.on_release)

    # Redraw the base image and all completed lines.
    def draw_image(self):
        with hold_canvas(self.canvas):
            self.canvas.put_image_data(self.image_rgba, 0, 0)
            self.canvas.stroke_style = "yellow"
            self.canvas.line_width = 2
            for (x0, y0, x1, y1) in self.lines:
                self.canvas.stroke_line(x0, y0, x1, y1)

    # Replace the displayed image while preserving drawn lines.
    def set_image(self, new_image_rgba):
        self.image_rgba = new_image_rgba
        self.draw_image()

    # Record the starting point of a new line.
    def on_press(self, x, y):
        self.start = (x, y)

    # Preview the line while the mouse is being dragged.
    def on_motion(self, x, y):
        if self.start is None:
            return
        x0, y0 = self.start
        with hold_canvas(self.canvas):
            self.canvas.put_image_data(self.image_rgba, 0, 0)
            self.canvas.stroke_style = "yellow"
            self.canvas.line_width = 2
            for (lx0, ly0, lx1, ly1) in self.lines:
                self.canvas.stroke_line(lx0, ly0, lx1, ly1)
            self.canvas.line_width = 1
            self.canvas.stroke_line(x0, y0, x, y)

    # Reserve a hook for future line-vector calculations.
    def calculate_vector(self, lines):
        if not lines:
            return
        # Placeholder - fill in with real vector/length calculations when needed.

    # Store a completed line and notify the profile plot.
    def on_release(self, x, y):
        if self.start is None:
            return
        x0, y0 = self.start
        self.lines.append((x0, y0, x, y))
        self.start = None
        self.draw_image()
        self.calculate_vector(self.lines)
        if self.on_line_drawn is not None:
            self.on_line_drawn()

    # Remove all lines from the current canvas.
    def clear_lines(self):
        self.lines.clear()
        self.draw_image()
        if self.on_line_drawn is not None:
            self.on_line_drawn()

# Display magnitude, each velocity direction, and MRA, selectable via a dropdown.
# Build and connect the interactive scan viewer for selected scans.
def run_function(scans):
    mag_idxs = [idx for idx, s in scans.items() if s["type"] == "M"]
    phase_idxs = [idx for idx, s in scans.items() if s["type"] == "P"]

    if not mag_idxs or not phase_idxs:
        print("Need at least one magnitude and one phase scan selected.")
        return

    mag_arr = scans[mag_idxs[0]]["array"]
    n_slices, n_time = mag_arr.shape[2], mag_arr.shape[4]
    v_mag = first_nonzero_vel(mag_arr)

    direction_panels = {}
    unlabeled = []
    for phase_idx in phase_idxs:
        for v, label in scans[phase_idx]["vel_directions"].items():
            if label:
                direction_panels.setdefault(label, (phase_idx, v))
            else:
                unlabeled.append((phase_idx, v))

    # --- DEBUG OVERRIDE: assume ordered X, Y, Z when no real labels were found ---
    DEBUG_ASSUME_XYZ_ORDER = True  # set False to go back to real label detection only

    if DEBUG_ASSUME_XYZ_ORDER and not direction_panels:
        all_vel_entries = []
        for phase_idx in phase_idxs:
            for v in scans[phase_idx]["vel_directions"].keys():
                all_vel_entries.append((phase_idx, v))
        all_vel_entries.sort(key=lambda pv: (pv[0], pv[1]))  # stable order: by phase_idx, then vel index

        for axis, entry in zip(["X", "Y", "Z"], all_vel_entries):
            direction_panels[axis] = entry
        unlabeled = all_vel_entries[3:]  # anything beyond the first 3 stays unlabeled
    # --- end debug override ---

    ordered_labels = [lbl for lbl in ["X", "Y", "Z"] if lbl in direction_panels]
    ordered_labels += [lbl for lbl in direction_panels if lbl not in ordered_labels]
    velocity_panels = [(lbl, direction_panels[lbl]) for lbl in ordered_labels]
    velocity_panels += [(f"Scan {p}, vel {v}", (p, v)) for p, v in unlabeled]

    panel_names = ["Magnitude"] + [lbl for lbl, _ in velocity_panels] + ["MRA"]

    slice_slider = widgets.IntSlider(min=0, max=n_slices - 1, value=n_slices // 2, description="Slice:", continuous_update=False)
    time_slider = widgets.IntSlider(min=1, max=n_time - 1, value=0, description="Time:", continuous_update=False)
    venc_box = widgets.FloatText(value=scans[velocity_panels[0][1][0]]["venc"] or 150.0, description="Venc (cm/s):")
    mra_threshold_box = widgets.FloatSlider(
        value=0.0,
        min=0.0,
        max=1.0,
        step=0.01,
        description="MRA threshold:",
        readout_format=".2f",
        continuous_update=False,
    )
    clear_lines_button = widgets.Button(description="Clear lines (current view)")
    view_dropdown = widgets.Dropdown(options=panel_names, value=panel_names[0], description="View:")

    # Render the velocity colorbar used beside velocity panels.
    def render_colorbar(venc, height, width=80):
        fig, ax = plt.subplots(figsize=(width / 100, height / 100), dpi=100)
        gradient = np.linspace(venc, -venc, height).reshape(-1, 1)
        ax.imshow(gradient, aspect="auto", cmap="seismic", vmin=-venc, vmax=venc,
                extent=[0, 1, -venc, venc])
        ax.set_xticks([])
        ax.yaxis.tick_right()
        ax.tick_params(labelsize=8)
        ax.set_ylabel("Velocity (cm/s)", rotation=270, labelpad=15, fontsize=8)
        ax.yaxis.set_label_position("right")
        fig.subplots_adjust(left=0.05, right=0.55, top=0.95, bottom=0.05)

        fig.canvas.draw()
        buf = np.asarray(fig.canvas.buffer_rgba())
        plt.close(fig)
        return buf

    # Create one time-frame velocity overlay for a selected direction.
    def make_velocity_image(phase_idx, v_phase):
        s, t, venc = slice_slider.value, time_slider.value, venc_box.value
        mag_img = mag_arr[:, :, s, v_mag, t]
        vel_img = phase_to_velocity(scans[phase_idx]["array"][:, :, s, v_phase, t], venc)
        return render_to_rgba(mag_img, vel_img, venc)

    # Return the selected velocity field across every time frame.
    def get_profile_velocity_over_time(view_name, s, venc):
        if view_name in direction_panels:
            phase_idx, v_phase = direction_panels[view_name]
            velocity = phase_to_velocity(scans[phase_idx]["array"][:, :, s, v_phase, :], venc)
            return velocity, view_name

        component_images = []
        for axis in ["X", "Y", "Z"]:
            if axis not in direction_panels:
                return None, None
            phase_idx, v_phase = direction_panels[axis]
            phase_pixels = scans[phase_idx]["array"][:, :, s, v_phase, :]
            component_images.append(phase_to_velocity(phase_pixels, venc))

        velocity_magnitude = np.sqrt(sum(component ** 2 for component in component_images))
        return velocity_magnitude, "Velocity magnitude"

    # Plot the spatially averaged velocity over time for each line.
    def update_profile(view_name):
        profile_output.clear_output(wait=True)
        drawer = drawers[view_name]
        if not drawer.lines:
            return

        velocity_over_time, profile_label = get_profile_velocity_over_time(
            view_name, slice_slider.value, venc_box.value
        )
        if velocity_over_time is None:
            with profile_output:
                print("Velocity profiles require X, Y, and Z direction data.")
            return

        height, width, n_profile_times = velocity_over_time.shape
        with profile_output:
            fig, ax = plt.subplots(figsize=(5, 4), dpi=100)
            for line_number, (x0, y0, x1, y1) in enumerate(drawer.lines, start=1):
                distance = np.hypot(x1 - x0, y1 - y0)
                sample_count = max(2, int(np.ceil(distance)) + 1)
                xs = np.linspace(x0, x1, sample_count)
                ys = np.linspace(y0, y1, sample_count)
                columns = np.clip(np.rint(xs).astype(int), 0, width - 1)
                rows = np.clip(np.rint(ys).astype(int), 0, height - 1)
                line_values = velocity_over_time[rows, columns, :]
                curve = np.nanmean(line_values, axis=0)
                ax.plot(np.arange(n_profile_times), curve, label=f"Line {line_number}")
            ax.axhline(0, color="black", linewidth=0.8)
            ax.set_title(f"{profile_label} over time")
            ax.set_xlabel("Time frame")
            ax.set_ylabel("Velocity (cm/s)")
            ax.grid(True, alpha=0.3)
            ax.legend()
            fig.tight_layout()
            display(fig)
            plt.close(fig)

    # Build one drawer per panel up front.
    drawers = {}
    colorbars = {}

    # Magnitude panel.
    mag_img0 = mag_arr[:, :, slice_slider.value, v_mag, time_slider.value]
    drawers["Magnitude"] = CanvasLineDrawer(render_mag_only(mag_img0))
    colorbars["Magnitude"] = None

    mag_img_height = mag_arr.shape[0]

    # Velocity direction panels.
    for label, (phase_idx, v_phase) in velocity_panels:
        drawers[label] = CanvasLineDrawer(make_velocity_image(phase_idx, v_phase))
        cbar_img = render_colorbar(venc_box.value, height=mag_img_height)
        cbar_canvas = Canvas(width=cbar_img.shape[1], height=cbar_img.shape[0])
        cbar_canvas.put_image_data(cbar_img, 0, 0)
        colorbars[label] = cbar_canvas

    # MRA panel.
    mra_img, missing = compute_mra(scans, mag_arr, v_mag, direction_panels, venc_box.value, slice_slider.value)

    if mra_img is None:
        placeholder = np.zeros((mag_arr.shape[0], mag_arr.shape[1], 4), dtype=np.uint8)
        drawers["MRA"] = CanvasLineDrawer(placeholder)
        drawers["MRA"]._mra_missing = missing
    else:
        drawers["MRA"] = CanvasLineDrawer(render_mra(mra_img, mra_threshold_box.value))
        drawers["MRA"]._mra_missing = None
    colorbars["MRA"] = None

    view_container = widgets.VBox([], layout=widgets.Layout(align_items="flex-start"))
    warning_label = widgets.HTML(value="")
    profile_output = widgets.Output(
        layout=widgets.Layout(width="520px", min_height="350px", border="1px solid lightgray")
    )

    # Refresh the scan and profile widgets for the active view.
    def refresh_view():

        name = view_dropdown.value
        drawer = drawers[name]

        if name == "MRA" and drawer._mra_missing:
            warning_label.value = f"<i>MRA requires X, Y, Z directions. Missing: {', '.join(drawer._mra_missing)}</i>"
            view_container.children = [warning_label]
            return

        scan_widgets = [drawer.canvas]
        if colorbars[name] is not None:
            scan_widgets.append(colorbars[name])
        view_container.children = [widgets.HBox(
            [widgets.HBox(scan_widgets, layout=widgets.Layout(align_items="flex-start")), profile_output],
            layout=widgets.Layout(align_items="flex-start")
        )]
        update_profile(name)

    # Recompute images and profiles when a viewer control changes.
    def update_all(change=None):
        s, t, venc = slice_slider.value, time_slider.value, venc_box.value

        drawers["Magnitude"].set_image(render_mag_only(mag_arr[:, :, s, v_mag, t]))

        for label, (phase_idx, v_phase) in velocity_panels:
            drawers[label].set_image(make_velocity_image(phase_idx, v_phase))
            cbar_img = render_colorbar(venc, height=mag_img_height)
            colorbars[label].put_image_data(cbar_img, 0, 0)

        mra_img, missing = compute_mra(scans, mag_arr, v_mag, direction_panels, venc, s)
        if mra_img is not None:
            drawers["MRA"].set_image(render_mra(mra_img, mra_threshold_box.value))
        drawers["MRA"]._mra_missing = missing

        refresh_view()

    # Refresh the display when the user selects another panel.
    def on_view_change(change):
        refresh_view()

    # Clear lines and their associated profile from the active panel.
    def on_clear_lines(b):
        drawers[view_dropdown.value].clear_lines()

    for drawer in drawers.values():
        drawer.on_line_drawn = lambda: update_profile(view_dropdown.value)

    slice_slider.observe(update_all, names="value")
    time_slider.observe(update_all, names="value")
    venc_box.observe(update_all, names="value")
    mra_threshold_box.observe(update_all, names="value")
    view_dropdown.observe(on_view_change, names="value")
    clear_lines_button.on_click(on_clear_lines)

    refresh_view()
    display(widgets.VBox([view_dropdown, slice_slider, time_slider, venc_box, mra_threshold_box, clear_lines_button, view_container]))

    return drawers

# Load the selected scans when the Run button is pressed.
# Load the scans selected in the table when Run is clicked.
def on_run_click(b):
    with output:
        clear_output()
        selected_indices = [i for i, cb in checkboxes.items() if cb.value]
        if not selected_indices:
            print("No scans selected.")
            return
        triage(df_display.loc[selected_indices])

run_button.on_click(on_run_click)

# Assemble and display the complete interactive viewer.
table = widgets.VBox([
    search_box,
    accordion_container,
    widgets.HTML(value="<b>Selected scans</b>"),
    selected_container,
    run_button,
    output,
])
display(table)

In [11]:
# Import DICOM, array, path, and Siemens metadata helpers for scan loading.
import pydicom
import numpy as np
from pathlib import Path
from nibabel.nicom import csareader

# Identify the dataframe column containing each scan folder.
SCAN_PATH_COL = "scan_path"

# Return the DICOM files in a stable order.
def get_dicom_files(folder):
    return sorted(Path(folder).glob("*.dcm"))

# Read one DICOM series and reconstruct its five-dimensional image array,
# capturing image type, venc, and per-direction labels in the same pass.
def load_dicom_series(folder):
    """Read a series once and return its array, image type, venc, and per-vel direction labels."""
    files = get_dicom_files(folder)
    if not files:
        raise ValueError(f"No DICOM files found in {folder}")

    # Map Siemens ICE_Dims positions to time, velocity, and slice axes.
    dim_time = 3 - 1
    dim_phase = 4 - 1
    dim_slice = 7 - 1

    images = []
    ice_vals_list = []
    skipped = []
    image_type = None
    venc = None
    vel_directions = {}  # vel index -> "X"/"Y"/"Z" label (or None if not found)

    # Read each file once, extracting metadata and pixels together.
    for file_index, fname in enumerate(files):
        dcm = pydicom.dcmread(fname)

        if file_index == 0:
            dcm_image_type = getattr(dcm, "ImageType", [])
            if "P" in dcm_image_type:
                image_type = "P"
            elif "M" in dcm_image_type:
                image_type = "M"

        csa = csareader.get_csa_header(dcm, "image")
        if csa is None:
            skipped.append(fname.name)
            continue

        if venc is None:
            venc_items = csa["tags"].get("FlowVenc", {}).get("items", [])
            if venc_items:
                venc = float(venc_items[0])

        ice_raw = csa["tags"]["ICE_Dims"]["items"][0]
        vals = [int(p.strip()) if p.strip().lstrip("-").isdigit() else 0 for p in ice_raw.split("_")]

        v = vals[dim_phase]
        if v not in vel_directions:
            dir_items = csa["tags"].get("FlowEncodingDirectionString", {}).get("items", [])
            vel_directions[v] = str(dir_items[0]).upper() if dir_items else None

        images.append(dcm.pixel_array.astype(float))
        ice_vals_list.append(vals)

    # Validate the decoded series before allocating the output array.
    if skipped:
        print(f"[warn] skipped {len(skipped)} file(s) with no CSA header in {folder}")
    if not images:
        raise ValueError(f"No valid CSA headers found in {folder}")
    if image_type is None:
        raise ValueError(f"Could not determine ImageType in {folder}")

    # Allocate the reconstructed array and place each image by ICE_Dims.
    ice_vals = np.asarray(ice_vals_list)
    n_slices = int(ice_vals[:, dim_slice].max()) + 1
    n_times = int(ice_vals[:, dim_time].max()) + 1
    n_vel = int(ice_vals[:, dim_phase].max()) + 1
    rows, cols = images[0].shape

    dcm_array = np.zeros((rows, cols, n_slices, n_vel, n_times), dtype=float)
    for image, vals in zip(images, ice_vals):
        s, t, v = vals[dim_slice], vals[dim_time], vals[dim_phase]
        dcm_array[:, :, s, v, t] = image

    return dcm_array, image_type, venc, vel_directions

# Find the first populated velocity channel efficiently.
def first_nonzero_vel(scan):
    nonzero_velocities = np.any(scan != 0, axis=(0, 1, 2, 4))
    nonzero_indices = np.flatnonzero(nonzero_velocities)
    return int(nonzero_indices[0]) if nonzero_indices.size else 0

# Cache decoded arrays so repeated Run-button clicks avoid network reads.
_scan_cache = {}

# Load selected scans, reusing cached data when possible.
def triage(selected_df):
    """Load each selected scan once, using cached arrays when available."""
    scans = {}
    total = len(selected_df)

    for position, (idx, row) in enumerate(selected_df.iterrows(), start=1):
        folder = str(row[SCAN_PATH_COL])
        if folder in _scan_cache:
            arr, img_type, venc, vel_directions = _scan_cache[folder]
            print(f"Using cached scan {position}/{total}: {Path(folder).name}")
        else:
            print(f"Loading scan {position}/{total}: {Path(folder).name}")
            arr, img_type, venc, vel_directions = load_dicom_series(folder)
            _scan_cache[folder] = (arr, img_type, venc, vel_directions)

        scans[idx] = {"array": arr, "type": img_type, "venc": venc, "folder": folder, "vel_directions": vel_directions}

    # Start the interactive viewer after all selected scans are ready.
    if not scans:
        print("No usable scans loaded.")
        return

    run_function(scans)